In [7]:
import polars as pl
import seaborn as sns
import datetime as dt

In [8]:
# Uso: Quando ler uma coluna de String é mais custosa que uma coluna numérica
# Obs.: Se todos os textos (conteúdos) forem diferentes não há o que fazer
# 2. No pandas:
#       .astype('Category')
# 3. No polars:
#       .cast(pl.Categorical)
# 4. Mais otimização:
#   Ideia: criar um dicionário para mapear de forma "global",
#          ou seja, qualquer tabela dentro desse arquivo
#          poderia usar esse mapeamento
pl.Categories()
# pl.enable_string_cache()

Categories()

In [9]:
# desativar a notação cientifica
pl.Config.set_fmt_float('full')
# definir numero maximo de casa pós vírguça
pl.Config.set_float_precision(2)

polars.config.Config

In [10]:
ARQUIVO = r'C:\Users\lucas.emoura\Documents\PastaLucas\aula01_UC2\aula02\dados_bronze\df_bf.parquet'

In [13]:
try:
    hora_inicio = dt.datetime.now()
    # criar um plano de execução
    df_bf_plano = pl.scan_parquet(ARQUIVO)
    df_lazy = df_bf_plano.select(
        [pl.col('NOME MUNICÍPIO'),
         pl.col('VALOR PARCELA'),
         pl.col('UF'),
         pl.col('MÊS REFERÊNCIA'),
         pl.col('MÊS COMPETÊNCIA')
         ]
    ).with_columns(
        (pl.col('MÊS REFERÊNCIA')
         .cast(pl.Utf8)
         .str.strptime(pl.Date, format='%Y%m')
         )
    )
    hora_fim = dt.datetime.now()
    print(f'Tempo de Execução: {hora_fim - hora_inicio}')
except Exception as e:
    print(f'Erro ao obter dados do parquet: {e}')

Tempo de Execução: 0:00:00.000253


In [15]:
COLUNA = 'VALOR PARCELA'

df_analise = df_lazy.select(
    pl.col(COLUNA).mean().alias('Média'),
    pl.col(COLUNA).median().alias('Mediana'),
    pl.col(COLUNA).std().alias('Desvio Padrão'),
    pl.col(COLUNA).skew().alias('Assimetria'),
    pl.col(COLUNA).kurtosis().alias('Curtose'),
    pl.col(COLUNA).min().alias('Mínimo'),
    pl.col(COLUNA).quantile(0.25).alias('Q1'),
    pl.col(COLUNA).quantile(0.75).alias('Q3'),
    pl.col(COLUNA).max().alias('Máximo')
).collect()
df_analise

Média,Mediana,Desvio Padrão,Assimetria,Curtose,Mínimo,Q1,Q3,Máximo
f64,f64,f64,f64,f64,f64,f64,f64,f64
668.26,650.00,190.28,0.89,4.67,25.00,600.00,750.00,3938.00
